# Apprentissage supervisé : méthodes non-paramétriques

# Table of contents
1. [Méthode des plus proches voisins](#part1)
1. [Arbes de décision](#part2)
1. [Forêts aléatoires](#part3)
1. [Boosting](#part4)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# %matplotlib inline
sns.set()

In [ ]:
from matplotlib import cm

def map_regions(clf, data=None, data_labels=None, num=200):
    """
        Map the regions f(x)=1…K of the classifier clf within the same range as the one
        of the data.
        Input:
            clf: classifier with a method predict
            data: input data (X)
            data_labels: data labels (y)
            num: discretization parameter
    """
    xmin, ymin = data.min(axis=0)
    xmax, ymax = data.max(axis=0)
    x, y = np.meshgrid(np.linspace(xmin, xmax, num), np.linspace(ymin, ymax))
    z = clf.predict(np.c_[x.ravel(), y.ravel()]).reshape(x.shape)
    zmin, zmax = z.min(), z.max()
    for icl, cl in enumerate(np.unique(data_labels)):
        plt.scatter(*data[data_labels==cl].T, label='Class {0:d}'.format(icl+1))
    plt.imshow(z, origin='lower', interpolation="nearest",
               extent=[xmin, xmax, ymin, ymax], cmap=cm.coolwarm,
              alpha=0.3)
    plt.axis('equal')
    minx, miny = data[:, 0].min(), data[:, 1].min()
    diffx, diffy = data[:, 0].max() - minx, data[:, 1].max() - miny
    plt.axis([minx - 0.1*diffx, minx + 1.1*diffx, miny - 0.1*diffy, miny + 1.1*diffy])
    plt.legend()

In [ ]:
# Dataset
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=500,
    n_classes=4,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1
)

# Méthode des plus proches voisins <a id="part1"></a>


<div class="alert alert-block alert-info">

Afficher les régions de classification obtenues par la méthode des plus proches voisins (paramètres par défaut).

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.neighbors import KNeighborsClassifier

five_near_neighbors_clf = KNeighborsClassifier()
# Fit the model
# Todo
five_near_neighbors_clf.fit(X, y)
# End todo

map_regions(five_near_neighbors_clf, X, y)

<div class="alert alert-block alert-info">

Faire de même en faisant varier le nombre de plus proches voisins.

<!-- <br> -->
</div>

In [ ]:
# Answer
plt.figure(figsize=(12, 20))
for it, k in enumerate([1, 2, 5, 10, 20, 30, 50, X.shape[0]]):
    # Build and fit the model
    # Todo
    knn_clf = KNeighborsClassifier(n_neighbors=k)
    knn_clf.fit(X, y)

    # End todo
    
    plt.subplot(4, 2, it+1)
    map_regions(knn_clf, X, y)
    plt.title("neighbors: {}, score: {}".format(k, knn_clf.score(X, y)))
plt.tight_layout()

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_pred, y, classes=None, normalize=False):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    title='Confusion matrix'
    cmap=plt.cm.Blues
    
    cm = confusion_matrix(y, y_pred)
    
    if classes is None:
        classes = np.unique(y)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = 'Normalized confusion matrix'
    else:
        title = 'Unnormalized confusion matrix'

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.grid()
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

<div class="alert alert-block alert-info">

Séparer le jeu de données en deux (utiliser `sklearn.model_selection.train_test_split`) et afficher la matrice de confusion obtenue sur le jeu de test.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8)

# Build and fit the model, store the predictions for X_test in y_pred
# Todo
final_knn_clf = KNeighborsClassifier(n_neighbors=5)
final_knn_clf.fit(X_train, y_train)
y_pred = final_knn_clf.predict(X_test)
# End todo

plot_confusion_matrix(y_pred, y_test)

# Arbes de décision <a id="part2"></a>


<div class="alert alert-block alert-info">

Comparer les régions de classification obtenues par les arbres de décision en faisant varier la profondeur maximale des arbres.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.tree import DecisionTreeClassifier

plt.figure(figsize=(12, 20))
for it, k in enumerate([1, 2, 3, 5, 10, 20, 30]):
    # Build and fit the model
    # Todo
    decision_tree_clf = DecisionTreeClassifier(max_depth=k)
    decision_tree_clf.fit(X, y)

    # End todo
    
    plt.subplot(4, 2, it+1)
    map_regions(decision_tree_clf, X, y)
    plt.title("maximal depth: {}, score: {}".format(k, decision_tree_clf.score(X, y)))
plt.tight_layout()

# Forêts aléatoires <a id="part3"></a>


<div class="alert alert-block alert-info">

Comparer les régions de classification obtenues par les forêts aléatoires en faisant varier le nombre d'arbres.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.ensemble import RandomForestClassifier

plt.figure(figsize=(12, 20))
for it, k in enumerate([1, 2, 3, 5, 10, 20, 30]):
    rf_clf = RandomForestClassifier(max_depth=3, n_estimators=k)
    # Fit the model
    # Todo
    rf_clf.fit(X, y)

    # End todo
    
    plt.subplot(4, 2, it+1)
    # Map the classification regions and add a title "number of estimators: XX, score: XX"
    # Todo
    map_regions(rf_clf, X, y)
    plt.title("number of estimators: {}, score: {}".format(k, rf_clf.score(X, y)))

    # End todo
plt.tight_layout()

# Boosting <a id="part4"></a>


<div class="alert alert-block alert-info">

Comparer les régions de classification obtenues par l'algorithme du *gradient boosting* en faisant varier le nombre d'arbres.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.ensemble import GradientBoostingClassifier

plt.figure(figsize=(12, 20))
for it, k in enumerate([1, 2, 3, 5, 10, 20, 30]):
    gradient_boosting_clf = GradientBoostingClassifier(max_depth=1, n_estimators=k)
    # Fit the model, map the regions and add a title
    # Todo
    gradient_boosting_clf.fit(X, y)
    
    plt.subplot(4, 2, it+1)
    map_regions(gradient_boosting_clf, X, y)
    plt.title("number of estimators: {}, score: {}".format(k, gradient_boosting_clf.score(X, y)))

    # End todo
plt.tight_layout()